# 05 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific **Clean** and **Noisy** champions from  (generated by Notebook 03).
2. Executes each champion **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Evaluates Clean Champions on clean settings () and Noisy Champions on noisy settings ().
4. Attaches IOH Analyzer context manager to output IOH performance files to .


In [1]:
import sys
import json
import hashlib
import shutil
import numpy as np
import pandas as pd
from pathlib import Path

# Add src to path
cwd = Path(".").resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core.config import DATA_DIR, PROJECT_ROOT
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection
from synthesis.execution import AlgorithmExecutor

# ── User Execution Controls & Selective Filters ──────────────────────────────
FORCE_REEVALUATE  = False   # Set True to bypass cache and re-evaluate all
FILTER_PROBLEMS   = None    # e.g., [8, 11] to evaluate specific problems, or None for all
FILTER_STRATEGIES = None    # e.g., ["baseline", "guided"] or None for all
FILTER_MODES      = None    # e.g., ["clean"], ["noisy"], or None for all
FILTER_DIMS       = None    # e.g., [2, 3], or None (uses all DIMS in database)
N_RUNS            = 10      # Number of independent benchmark runs per config
TIMEOUT_SECONDS   = 30.0    # Per-run execution timeout in seconds

# ── Dynamic Experiment Parameters Extracted Purely from Database ──────────────
CHAMPIONS_PATH = DATA_DIR / "champions.json"
IOH_LOGS_DIR   = DATA_DIR / "ioh_logs"

with get_db_connection() as conn:
    df_exp_meta = pd.read_sql_query(
        "SELECT DISTINCT dim, budget, noise_std, problem_id FROM experiments WHERE status = 'completed'",
        conn
    )

if df_exp_meta.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

DIMS   = sorted(df_exp_meta["dim"].dropna().astype(int).unique().tolist())
BUDGET = int(df_exp_meta["budget"].dropna().max())

# Apply FILTER_DIMS if specified
if FILTER_DIMS is not None:
    DIMS = [d for d in DIMS if d in FILTER_DIMS]

print(f"🎯 Dynamic parameters loaded from database:")
print(f"  • Evaluated Dimensions: {DIMS}")
print(f"  • Benchmark Budget:     {BUDGET}")
print(f"  • Runs per config:      {N_RUNS}")
print(f"  • Champions JSON:       {CHAMPIONS_PATH}")
print(f"  • IOH Logs Output:      {IOH_LOGS_DIR}")
print(f"  • Force Re-evaluate:    {FORCE_REEVALUATE}")
if FILTER_PROBLEMS:   print(f"  • Filter Problems:      f{FILTER_PROBLEMS}")
if FILTER_STRATEGIES: print(f"  • Filter Strategies:    {FILTER_STRATEGIES}")
if FILTER_MODES:      print(f"  • Filter Modes:         {FILTER_MODES}")


🎯 Dynamic parameters loaded from database:
  • Evaluated Dimensions: [2, 3, 5]
  • Benchmark Budget:     1000000
  • Runs per config:      10
  • Champions JSON:       /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/champions.json
  • IOH Logs Output:      /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/ioh_logs
  • Force Re-evaluate:    False


## 1. Load Champions JSON

In [2]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 03 first.')

with open(CHAMPIONS_PATH, 'r', encoding='utf-8') as f:
    champions_raw = json.load(f)

# Support both nested {model: {key: info}} and legacy flat {key: info}
champions_flat = {}
for k, v in champions_raw.items():
    if isinstance(v, dict) and 'code_path' in v:
        champions_flat[k] = v
    elif isinstance(v, dict):
        for sub_k, sub_v in v.items():
            champions_flat[f'{k}/{sub_k}'] = sub_v

print(f'Loaded {len(champions_flat)} champion configuration(s) across {len(champions_raw)} model category(ies):')
for model_key, model_dict in champions_raw.items():
    if isinstance(model_dict, dict) and 'code_path' not in model_dict:
        print(f'  • {model_key}: {len(model_dict)} champions')


Loaded 128 champion configuration(s) across 2 model category(ies):
  • qwen2.5-coder-14b-instruct-q4_k_m.gguf: 108 champions
  • qwen2.5-coder-7b-instruct-q4_k_m.gguf: 20 champions


## 2. Execute Champion Evaluation Benchmark

In [3]:
import shutil
import hashlib
from infra.problems import ProblemAnalyzer
from synthesis.execution import AlgorithmExecutor
import ioh
import re

executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

skipped = []
evaluated = []

def _compute_code_hash(code_content: str) -> str:
    """Compute a SHA-256 hash of the champion code for cache invalidation."""
    return hashlib.sha256(code_content.encode("utf-8")).hexdigest()

def _read_provenance(prov_path: Path) -> dict | None:
    """Read provenance.json, returning None if missing or malformed."""
    try:
        return json.loads(prov_path.read_text(encoding="utf-8"))
    except Exception:
        return None

def _write_provenance(prov_path: Path, info: dict, dim: int, noise_std: float, code_hash: str, med_err: float) -> None:
    """Write provenance metadata alongside the IOH log folder."""
    prov = {
        "experiment_id":     int(info.get("experiment_id", -1)),
        "iteration_id":      int(info.get("iteration_id", -1)) if "iteration_id" in info else None,
        "algorithm_name":    info["algorithm_name"],
        "code_path":         info["code_path"],
        "code_hash":         code_hash,
        "problem_id":        int(info["problem_id"]),
        "dim":               dim,
        "noise_std":         noise_std,
        "prompt_strategy":   info.get("prompt_strategy", "baseline"),
        "llm_name":          info.get("llm_name", ""),
        "mode":              info.get("mode", "all"),
        "n_runs":            N_RUNS,
        "median_clean_error": float(med_err) if not np.isinf(med_err) else None,
        "evaluated_at":      pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding="utf-8")

print(f"=== Starting Evaluation of {len(champions_flat)} Dimension-Specific Champions ===")

for key, info in champions_flat.items():
    p_id       = int(info["problem_id"])
    dim        = int(info["dim"])
    mode       = info.get("mode", "all").lower()
    strat      = info.get("prompt_strategy", "baseline").lower()
    llm_name   = info.get("llm_name", key.split("/")[0])
    noise_std  = float(info.get("noise_std", 0.0))
    exp_id     = int(info.get("experiment_id", -1))
    algo_name  = info["algorithm_name"]
    clean_key  = key.split("/")[-1]

    # User Filters
    if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS: continue
    if FILTER_STRATEGIES and strat not in FILTER_STRATEGIES: continue
    if FILTER_MODES and mode not in FILTER_MODES: continue
    if FILTER_DIMS and dim not in FILTER_DIMS: continue

    code_file = PROJECT_ROOT / info["code_path"] if not Path(info["code_path"]).is_absolute() else Path(info["code_path"])
    if not code_file.exists():
        print(f"[WARN] Code file for {clean_key} not found at {code_file}. Skipping.")
        continue

    code_content = code_file.read_text(encoding="utf-8")
    code_hash    = _compute_code_hash(code_content)

    # Destination path in data/ioh_logs
    out_dir = IOH_LOGS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
    folder_name = f"llamea_{strat}_{mode}" if "14b" in llm_name.lower() else f"llamea_7b_{strat}_{mode}"
    target_log_folder = out_dir / folder_name
    prov_path = target_log_folder / "provenance.json"

    # Provenance Check (cache validation)
    if not FORCE_REEVALUATE and target_log_folder.exists():
        prov = _read_provenance(prov_path)
        dat_files = [f for f in target_log_folder.glob("**/*.dat") if f.stat().st_size > 0]
        if prov and prov.get("code_hash") == code_hash and len(dat_files) > 0:
            skipped.append(clean_key)
            continue

    print(f"⚡ Evaluating [{llm_name}] {clean_key} ({algo_name}, Exp #{exp_id}) for {N_RUNS} runs...")
    # Delete existing target log folder if re-evaluating to prevent duplicate -1, -2 suffixes
    if target_log_folder.exists():
        shutil.rmtree(target_log_folder, ignore_errors=True)
    target_log_folder.mkdir(parents=True, exist_ok=True)

    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    run_errors = []

    # Initialize logger ONCE for all N_RUNS of this condition
    logger = ioh.logger.Analyzer(
        root=str(out_dir),
        folder_name=folder_name,
        algorithm_name=f"LLaMEA-{llm_name}/{strat}",
        store_positions=False
    )

    for run_idx in range(1, N_RUNS + 1):
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=run_idx,
            noise_strategy=noise_strat,
        )
        problem.attach_logger(logger)

        try:
            x_opt, f_opt = executor.execute_algorithm(
                code=code_content,
                name=algo_name,
                dim=dim,
                problem=problem,
                budget=BUDGET
            )
            clean_prob = BBOBProblem(problem_id=p_id, dim=dim, instance_id=run_idx, noise_strategy=NoNoiseStrategy())
            final_err = clean_prob(x_opt) if x_opt is not None else float(f_opt)
            run_errors.append(final_err)
        except Exception as e:
            print(f"   ❌ Run {run_idx}/{N_RUNS} failed: {e}")
            run_errors.append(float("inf"))
        finally:
            if hasattr(problem, "clean_problem") and hasattr(problem.clean_problem, "detach_logger"):
                problem.clean_problem.detach_logger()

    del logger
    med_err = np.median(run_errors) if run_errors else float("inf")
    _write_provenance(prov_path, info, dim, noise_std, code_hash, med_err)
    evaluated.append(clean_key)
    print(f"   ✅ Completed {len(run_errors)}/{N_RUNS} runs (Median Error: {med_err:.4e})")

print("\n" + "="*70)
print(f"🎯 Evaluation Summary: {len(evaluated)} evaluated, {len(skipped)} skipped (cached)")
print("="*70)


=== Starting Evaluation of 128 Dimension-Specific Champions ===
⚡ Evaluating [qwen2.5-coder-14b-instruct-q4_k_m.gguf] f11_3D_clean_guided (EnhancedDEOptimizer, Exp #99) for 10 runs...
   ✅ Completed 10/10 runs (Median Error: -4.4450e+00)
⚡ Evaluating [qwen2.5-coder-14b-instruct-q4_k_m.gguf] f11_3D_noisy_guided (NoisyOptimizer, Exp #116) for 10 runs...
   ✅ Completed 10/10 runs (Median Error: -4.4441e+00)
⚡ Evaluating [qwen2.5-coder-14b-instruct-q4_k_m.gguf] f11_3D_clean_thinking (EnhancedExploitationGuidedOptimizer, Exp #89) for 10 runs...
   ✅ Completed 10/10 runs (Median Error: 3.2351e+02)
⚡ Evaluating [qwen2.5-coder-14b-instruct-q4_k_m.gguf] f11_3D_noisy_thinking (ImprovedNoisyOptimization, Exp #104) for 10 runs...
   ✅ Completed 10/10 runs (Median Error: -4.2997e+00)
⚡ Evaluating [qwen2.5-coder-14b-instruct-q4_k_m.gguf] f11_3D_noisy_vectorization (NoiseResilientOptimization, Exp #109) for 10 runs...
   ✅ Completed 10/10 runs (Median Error: -3.4631e+00)
⚡ Evaluating [qwen2.5-coder-1

KeyboardInterrupt: 